In [0]:
from pyspark.sql import functions as F

BRONZE = "olist_lakehouse.bronze"
INCREMENTAL_PATH = "/Volumes/olist_lakehouse/landing/olist/incremental"
CHECKPOINT_BASE = "/Volumes/olist_lakehouse/landing/checkpoints"

ENTITIES = {
    "orders": "orders",
    "order_items": "order_items",
    "order_payments": "order_payments",
    "customers": "customers",
}

In [0]:
def ingest_incremental(entity: str, table: str) -> int:

    full_table_name = f"{BRONZE}.{table}"
    checkpoint = f"{CHECKPOINT_BASE}/bronze_{table}"
    rows_before = spark.table(full_table_name).count()

    query = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{checkpoint}/_schema")
        .load(f"{INCREMENTAL_PATH}/{entity}")
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
        .writeStream
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(full_table_name)
    )

    query.awaitTermination() 
    return spark.table(full_table_name).count() - rows_before

In [0]:
for entity, table in ENTITIES.items():
    new_rows = ingest_incremental(entity, table)
    print(f"{BRONZE}.{table:<16} +{new_rows:,} new records")

In [0]:
%sql
SELECT order_id, order_status, order_purchase_timestamp, _source_file, _ingested_at, _rescued_data
FROM olist_lakehouse.bronze.orders
WHERE _source_file LIKE '%/incremental/%'
ORDER BY _ingested_at DESC
LIMIT 20

In [0]:
%sql
DESCRIBE HISTORY olist_lakehouse.silver.orders

In [0]:
%sql
SELECT order_id, order_status, order_purchase_timestamp, _source_file
FROM olist_lakehouse.silver.orders
WHERE order_purchase_timestamp IS NULL